### NexaCorp Chatbot

In [2]:
# import necessary libraries
import re
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [3]:
# load environment variables
load_dotenv()

True

Load the document

In [5]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
try:
    loader = UnstructuredWordDocumentLoader(
        file_path,
        mode="elements"                                 # breaks document into structured chunks based on layout
    )
    docs = loader.load()
except Exception as e:
    raise RuntimeError(f"Failed to load document: {file_path}") from e

Remove unwanted data like header, footer, table of contents

In [6]:
# use regular expressions to remove header/footer noise
def is_header_footer(text):
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",
        r"Internal Restricted",
        r"Not for External Distribution",
        r"©\s*20\d{2}",
        r"CONFIDENTIAL",
        r"Policy.*Handbook"
    ]
    # checks if patterns exist anywhere in the text and returns True if atleast one pattern matches
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)

# uses regular expression to check if line belongs to TOC or not
def is_toc_line(text):
    text = text.strip()
    return bool(re.search(
        # check for Part I/3.2+Company Overview & Mission+5
        r"^(Part\s+[IVXLC]+|[\d]+\.[\d]+)\s+.*\s+\d+$",
        text,
        re.IGNORECASE
    ))

# clean the documents by removing unwanted data
def clean_documents(docs):
    cleaned_docs = []
    skip_toc = False
    removed = 0
    for doc in docs:
        text = doc.page_content.strip()
        lower = text.lower()
        category = doc.metadata.get("category", "")
        # Remove headers/footers using metadata
        if category in ["Header", "Footer"]:
            removed += 1
            continue
        # Regex-based, if metadata failed
        if is_header_footer(text):
            removed += 1
            continue
        # TOC detection
        if "table of contents" in lower:
            skip_toc = True
            continue
        # if in TOC section and line belongs to TOC -> remove
        if skip_toc:
            if is_toc_line(text) or len(text.split()) < 3:
                continue
            else:
                skip_toc = False
        # separate table and text metadata
        if category == "Table":
            doc.metadata["type"] = "table"
        else:
            doc.metadata["type"] = "text"
        # add cleaned documents to list
        cleaned_docs.append(doc)
        
    print(f"Removed {removed} noisy chunks")
    return cleaned_docs

In [7]:
docs_clean = clean_documents(docs)

Removed 21 noisy chunks


In [8]:
# check if cleaning worked or not
print(docs[10])
print(docs_clean[10])

page_content='Tel: +1 (415) 700-9000  |  hr@nexacorp.com  |  www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-05-01T11:30:41', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752', 'type': 'text'}
page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5', 'type': 'text'}


Create structured docs with section, sub-section, content 

In [9]:
structured_docs = []
current_section = "General"                             # default section name
current_subsection = ""                                 # track subsection separately

for doc in docs_clean:
    text = doc.page_content.strip()                     # page content without extra spaces
    category = doc.metadata.get("category", "")         # to get type of element like Title, Text etc
    if not text:                                        # skip empty chunks
        continue
    # if chunk is Title update the section/subsection
    if category == "Title":                             
        # detect subsection like 1.2, 3.4.1 etc
        if re.match(r"\d+\.\d+", text):
            current_subsection = text
        else:
            current_section = text
            current_subsection = ""                     # reset subsection when new section starts
        # DON'T skip → keep title in content for better context
        structured_docs.append({
            "section": current_section,
            "subsection": current_subsection,
            "content": text,
            "type": "title"
        })
        continue   
    # add section and content to structured_docs
    structured_docs.append({                            
        "section": current_section,
        "subsection": current_subsection,
        "content": text,
        "type": category                                
    })

In [10]:
print(len(structured_docs))

431


Merge the documents to create parent docs

In [11]:
# merge content (by section + subsection + content)
merged_docs = {}

for item in structured_docs:
    section = item["section"]
    subsection = item.get("subsection", "")
    text = item["content"]
    key = f"{section} > {subsection}"                   # unique key for grouping
    if key not in merged_docs:
        merged_docs[key] = {
            "section": section,
            "subsection": subsection,
            "content": ""
        }
    merged_docs[key]["content"] += "\n" + text          # merge content

In [13]:
print(len(merged_docs))

150


In [75]:
# text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      
    chunk_overlap=100,
    separators=["\n\n", "\n", "."]  
)

final_chunks = []

for key, item in merged_docs.items():
    section = item["section"]                           # extract the section and text of merged doc
    subsection = item["subsection"]
    text = item["content"]

    chunks = splitter.split_text(text)                  # split text into chunks

    for i, chunk in enumerate(chunks):
        final_chunks.append({                           # store content and metadata of final chunk 
            "content": chunk,
            "section": section,
            "subsection": subsection,
            "chunk_id": f"{key}_{i}"                    # unique id per chunk
        })

In [ ]:
# convert chunks to LangChain Documents
documents = []                                      

for chunk in final_chunks:
    section = chunk.get("section", "")
    subsection = chunk.get("subsection", "")
    content = chunk["content"]

    # prepend context to improve embeddings
    final_content = f"{section}\n{subsection}\n{content}"

    documents.append(
        Document(
            page_content=final_content,
            metadata={
                "section": section,
                "subsection": subsection,
                "chunk_id": chunk.get("chunk_id", ""),
                "source": "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx",
                "type": chunk.get("type", "text")
            }
        )
    )

In [77]:
print(len(documents))

153


In [78]:
# create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1558.77it/s]


In [ ]:
# create FAISS vector store
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
        documents=documents,
        embedding=embeddings
    )

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",  
    search_kwargs={
        "k": 5 
    }
)

In [83]:
query = "Explain company values and organisational structure together"
docs = retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"\n--- Chunk {i+1} ---")
    print("Section:", doc.metadata.get("section"))
    print("Content:", doc.page_content)


--- Chunk 1 ---
Section: 1. Company Overview & Mission
Content: 1. Company Overview & Mission
1.2 Vision, Mission & Core Values
Value What It Means in Practice Integrity First We act honestly and transparently even when it is difficult. We do not misrepresent data, timelines, or capabilities to clients, colleagues, or regulators. Customer Obsession Every decision is evaluated through the lens of long-term customer value. Short-term revenue that compromises customer trust is never acceptable. Bold Innovation We reward calculated risk-taking and intelligent failure. Bureaucracy that impedes experimentation must be challenged through proper channels. Inclusive Excellence Diversity is not a compliance exercise — it is a competitive advantage. We build teams that reflect the global communities we serve. Ownership Mindset Every employee takes responsibility for outcomes, not just activities

--- Chunk 2 ---
Section: 2. Code of Business Conduct & Ethics
Content: 2. Code of Business Conduct &